# 실습 C: AI 비서 대화 시스템

챗봇이 이전 대화를 기억하려면 어떻게 해야 할까요? LangChain은 다양한 메모리(Memory) 방식을 제공해요. 각 방식은 **저장 범위**, **토큰 소비**, **영속성**이 서로 달라요.

이 실습에서는 네 가지 메모리 패턴을 직접 구현하고 차이를 비교해볼 거예요.

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:
- `ConversationBufferMemory`로 전체 대화 기록을 유지하는 챗봇을 만들 수 있어요
- `BufferWindowMemory`와 `ConversationSummaryMemory`의 트레이드오프를 설명할 수 있어요
- `RunnableWithMessageHistory` + SQLite로 세션 간 대화를 영속적으로 저장할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 Ch04 Memory에서 다룬 메모리 클래스 개요와 Ch05 Advanced-LCEL의 `RunnableWithMessageHistory`를 이해하고 있어야 해요.

---

아래 다이어그램은 네 가지 메모리 방식의 동작 구조를 보여줘요.

```mermaid
graph TD
    Q{얼마나 오래<br/>기억해야 하나?}:::process

    Q -->|전체 기억| A[ConversationBufferMemory<br/>모든 대화 저장]:::output
    Q -->|최근만 기억| B[BufferWindowMemory<br/>최근 k턴만 유지]:::output
    Q -->|압축해서 기억| C[ConversationSummaryMemory<br/>LLM으로 요약 저장]:::output
    Q -->|영구 기억| D[SQLite + RunnableWithMessageHistory<br/>DB에 영속 저장]:::storage

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## 환경 설정

실습에 필요한 패키지를 불러오고 API 키를 로드해요.

In [1]:
# ---------------------------------------------------
# 공통 패키지 임포트 및 환경 변수 로드
# ---------------------------------------------------
from dotenv import load_dotenv

# .env 파일에서 OPENAI_API_KEY를 읽어요
load_dotenv()

True

In [2]:
# ---------------------------------------------------
# 실습 전체에서 공유할 유틸리티 함수 정의
# ---------------------------------------------------
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def build_chat_chain(memory):
    """메모리 객체를 받아 대화 체인을 반환하는 헬퍼 함수예요.

    Args:
        memory: LangChain 메모리 객체 (memory_key='chat_history' 필요)

    Returns:
        invoke(query) 를 지원하는 체인 객체
    """
    llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "당신은 친절한 AI 비서예요. 이전 대화를 참고해 답변해요."),
            # chat_history 자리표시자(Placeholder): 메모리에서 꺼낸 대화 기록이 삽입돼요
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )

    # RunnablePassthrough.assign: input 그대로 통과하면서 chat_history 키를 추가해요
    chain = (
        RunnablePassthrough.assign(
            chat_history=RunnableLambda(memory.load_memory_variables)
            | itemgetter("chat_history")
        )
        | prompt
        | llm
        | StrOutputParser()
    )

    return chain


def chat(chain, memory, query: str) -> str:
    """체인으로 답변을 생성하고 메모리에 저장하는 함수예요."""
    response = chain.invoke({"input": query})
    # save_context: 사용자 입력과 AI 응답 한 쌍을 메모리에 기록해요
    memory.save_context({"human": query}, {"ai": response})
    return response


def print_history(memory):
    """현재 메모리에 저장된 대화 기록을 출력하는 함수예요."""
    history = memory.load_memory_variables({})["chat_history"]
    print(f"=== 저장된 대화 기록 (총 {len(history)}개 메시지) ===")
    for msg in history:
        role = "사용자" if msg.type == "human" else "AI"
        # 내용이 길면 앞 80자만 표시해요
        content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
        print(f"[{role}] {content}")
    print()

---

## 실습 1: 기억하는 챗봇

`ConversationBufferMemory`(대화 버퍼 메모리)는 발생한 모든 대화를 그대로 저장해요. 대화가 누적될수록 메모리 크기가 커지지만, 초반 발언까지 정확히 참조할 수 있어요.

아래 다이어그램은 버퍼 메모리의 동작 방식을 보여줘요.

```mermaid
sequenceDiagram
    participant U as 사용자
    participant C as ConversationChain
    participant M as ConversationBufferMemory
    participant L as LLM

    U->>C: 질문 전달
    C->>M: load_memory_variables() 호출
    M-->>C: 전체 대화 기록 반환
    C->>L: 프롬프트 (시스템 + 전체 기록 + 질문) 전달
    L-->>C: 응답 생성
    C->>M: save_context() 로 응답 저장
    C-->>U: 응답 반환
```

**과제**: AI 비서 역할의 챗봇을 만들어요. 사용자가 자신의 이름과 관심사를 소개한 뒤, 5턴 대화 후 챗봇이 초반 발언을 기억하는지 확인해요.

**요구사항**:
- `ConversationBufferMemory`를 생성하고 `return_messages=True`로 설정해요
- `build_chat_chain(memory)`으로 체인을 구성해요
- 5턴 이상 대화 후 "내 이름이 뭐야?"를 물어봐요

**힌트**: `memory_key="chat_history"` 를 설정해야 프롬프트의 `MessagesPlaceholder`와 연결돼요

In [6]:
# ============================================================
# TODO: ConversationBufferMemory 기반 챗봇 구성
# 힌트: ConversationBufferMemory(return_messages=True, memory_key="chat_history")
# 예상 결과: 5턴 대화 후 이름 질문에 첫 번째 발언 내용을 정확히 답해야 해요
# ============================================================

from langchain.memory import ConversationBufferMemory

# 여기에 코드를 작성하세요
memory_buf = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

### 풀이: 실습 1

단계별로 메모리 설정, 체인 구성, 5턴 대화를 구현해볼게요.

In [7]:
# ---------------------------------------------------
# 실습 1 풀이: ConversationBufferMemory 챗봇
# ---------------------------------------------------
memory_buf.save_context(
    inputs={"human": "안녕하세요? 여행 계획을 세우고 싶어요. 제주도 여행을 추천해주세요"},
    outputs={"ai": "안녕하세요! 제주도 여행 계획을 도와드리겠습니다. 몇 박 며칠 일정을 생각하고 계신가요?"}
)

In [8]:
# ---------------------------------------------------
# 메모리에 저장된 전체 대화 기록 확인
# ---------------------------------------------------
# ConversationBufferMemory는 모든 대화를 그대로 보관해요
print_history(memory_buf)

=== 저장된 대화 기록 (총 2개 메시지) ===
[사용자] 안녕하세요? 여행 계획을 세우고 싶어요. 제주도 여행을 추천해주세요
[AI] 안녕하세요! 제주도 여행 계획을 도와드리겠습니다. 몇 박 며칠 일정을 생각하고 계신가요?



> **실무 팁**: `ConversationBufferMemory`는 대화가 길어질수록 LLM에 전달하는 토큰이 선형으로 증가해요. 짧은 세션에는 적합하지만, 수십 턴 이상의 대화에는 아래 실습에서 다루는 윈도우 방식이나 요약 방식이 더 효율적이에요.

---

## 실습 2: 효율적 메모리 관리

대화가 길어지면 버퍼 메모리는 토큰 낭비가 심해요. 두 가지 제한 방식을 비교해볼게요.

- `ConversationBufferWindowMemory(k=3)`: **최근 k턴**만 유지해요
- `ConversationTokenBufferMemory(max_token_limit=200)`: **토큰 수** 기준으로 오래된 대화를 제거해요

아래 다이어그램은 두 방식의 차이를 보여줘요.

```mermaid
graph LR
    A[전체 대화 기록]:::input

    A -->|k=3 윈도우| B[최근 3턴만<br/>유지]:::process
    A -->|토큰 200 제한| C[200 토큰 범위 내<br/>최신 대화 유지]:::process

    B --> D[일정 메시지 수<br/>보장]:::output
    C --> E[일정 토큰 비용<br/>보장]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

**과제**: 동일한 대화 시나리오(6턴)를 두 메모리에 저장한 뒤, 각각 보관 중인 내용을 비교해요.

**요구사항**:
- `ConversationBufferWindowMemory(k=3, ...)`와<br/>`ConversationTokenBufferMemory(max_token_limit=200, ...)`를 각각 생성해요
- 같은 6턴 대화를 두 메모리에 `save_context`로 직접 저장해요
- `print_history`로 결과를 나란히 출력해 차이를 확인해요

**힌트**: `ConversationTokenBufferMemory`는 `llm` 파라미터로 토크나이저 기준 모델을 받아요

In [12]:
# ============================================================
# TODO: BufferWindowMemory vs TokenBufferMemory 비교 실험
# 힌트: 두 메모리를 생성하고 동일한 6턴 대화를 save_context로 저장한 뒤 비교해요
# 예상 결과: Window는 마지막 3턴, Token은 200토큰 내 최신 대화만 남아야 해요
# ============================================================

# 여기에 코드를 작성하세요
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferWindowMemory, ConversationTokenBufferMemory

llm = ChatOpenAI(model_name="gpt-4o-mini")

memory_window = ConversationBufferWindowMemory(k=3, return_messages=True, memory_key="chat_history")
memory_token = ConversationTokenBufferMemory(llm=llm, max_token_limit=200, return_messages=True, memory_key="chat_history")

conversations = [
    ("이번 주말에 친구들 2명과 함께 제주도로 여행을 가기로 했어요.", 
    "와, 제주도 여행이라니 정말 즐겁겠네요! 친구분들과 어떤 테마의 여행을 계획 중이신가요?"),
    
    ("저희 셋 다 운동을 좋아해서 한라산에 등산하러 가기로 결정했어요.", 
    "활동적인 여행이 되겠네요! 한라산은 코스마다 난이도가 다른데, 초보자용과 숙련자용 중 어떤 코스를 원하시나요?"),
    
    ("저희는 등산 초보라서 경치가 예쁘면서도 너무 힘들지 않은 짧은 코스가 좋겠어요.", 
    "그렇다면 '영실 코스'를 강력 추천드려요! 비교적 짧지만 기암괴석과 제주의 풍경이 한눈에 들어와서 정말 아름답거든요."),
    
    ("영실 코스 좋네요! 등산할 때 특별히 챙겨야 할 준비물이 있을까요?", 
    "산 위는 바람이 많이 불고 기온이 낮으니 가벼운 바람막이 점퍼와 충분한 물, 그리고 간편한 간식은 꼭 챙기세요."),
    
    ("준비 완료! 그런데 등산하고 내려오면 너무 배고플 것 같은데 근처에 맛집이 있나요?", 
    "등산 후에는 역시 제주 흑돼지죠! 영실 코스에서 차로 가까운 중문 단지 쪽에 유명한 흑돼지 전문점들이 아주 많아요."),
    
    ("흑돼지라니 벌써 군침이 도네요! 그중에서도 가장 인기 있는 식당 한 곳만 찝어줄래요?", 
    "요즘 가장 핫한 곳은 '숙성도'나 '칠돈가'예요. 웨이팅이 길 수 있으니 등산 마치기 전에 미리 원격 줄서기를 해두는 걸 잊지 마세요!")
]

for human_msg, ai_reply in conversations:
    memory_window.save_context({"human": human_msg}, {"ai": ai_reply})
    memory_token.save_context({"human": human_msg}, {"ai": ai_reply})



### 풀이: 실습 2

두 메모리를 나란히 구성하고 동일한 6턴 대화를 저장한 뒤 결과를 비교해볼게요.

In [ ]:
# ---------------------------------------------------
# 실습 2 풀이: 두 메모리 생성 및 비교
# ---------------------------------------------------
pass

In [13]:
# ---------------------------------------------------
# 두 메모리 비교: 보존된 대화 내용 출력
# ---------------------------------------------------
print("[ BufferWindowMemory (k=3) ]")
print_history(memory_window)

print("[ ConversationTokenBufferMemory (max_token_limit=200) ]")
print_history(memory_token)

[ BufferWindowMemory (k=3) ]
=== 저장된 대화 기록 (총 6개 메시지) ===
[사용자] 영실 코스 좋네요! 등산할 때 특별히 챙겨야 할 준비물이 있을까요?
[AI] 산 위는 바람이 많이 불고 기온이 낮으니 가벼운 바람막이 점퍼와 충분한 물, 그리고 간편한 간식은 꼭 챙기세요.
[사용자] 준비 완료! 그런데 등산하고 내려오면 너무 배고플 것 같은데 근처에 맛집이 있나요?
[AI] 등산 후에는 역시 제주 흑돼지죠! 영실 코스에서 차로 가까운 중문 단지 쪽에 유명한 흑돼지 전문점들이 아주 많아요.
[사용자] 흑돼지라니 벌써 군침이 도네요! 그중에서도 가장 인기 있는 식당 한 곳만 찝어줄래요?
[AI] 요즘 가장 핫한 곳은 '숙성도'나 '칠돈가'예요. 웨이팅이 길 수 있으니 등산 마치기 전에 미리 원격 줄서기를 해두는 걸 잊지 마세요!

[ ConversationTokenBufferMemory (max_token_limit=200) ]
=== 저장된 대화 기록 (총 4개 메시지) ===
[사용자] 준비 완료! 그런데 등산하고 내려오면 너무 배고플 것 같은데 근처에 맛집이 있나요?
[AI] 등산 후에는 역시 제주 흑돼지죠! 영실 코스에서 차로 가까운 중문 단지 쪽에 유명한 흑돼지 전문점들이 아주 많아요.
[사용자] 흑돼지라니 벌써 군침이 도네요! 그중에서도 가장 인기 있는 식당 한 곳만 찝어줄래요?
[AI] 요즘 가장 핫한 곳은 '숙성도'나 '칠돈가'예요. 웨이팅이 길 수 있으니 등산 마치기 전에 미리 원격 줄서기를 해두는 걸 잊지 마세요!



> **주의**: `k=3`은 *턴* 기준이라 메시지 수로는 최대 6개(Human 3 + AI 3)를 유지해요. 반면 토큰 기준 메모리는 긴 문장이 섞이면 더 적은 턴을 보관하기도 해요.

---

## 실습 3: 요약 메모리 챗봇

`ConversationSummaryMemory`(대화 요약 메모리)는 오래된 대화를 LLM으로 **요약**해서 보관해요. 긴 대화에서도 토큰 사용량을 일정 수준으로 유지할 수 있어요.

아래 다이어그램은 요약 메모리의 동작 흐름을 보여줘요.

```mermaid
sequenceDiagram
    participant U as 사용자
    participant M as SummaryMemory
    participant S as 요약 LLM
    participant L as 응답 LLM

    U->>M: 새 대화 추가 (save_context)
    M->>S: 기존 요약 + 새 대화를 요약 요청
    S-->>M: 갱신된 요약 저장
    U->>L: 다음 질문 전달
    M-->>L: 요약본 제공 (전체 기록 대신)
    L-->>U: 응답 생성
```

**과제**: 10턴 이상 대화를 나누고 메모리에 저장된 요약 내용을 확인해요.

**요구사항**:
- `ConversationSummaryMemory(llm=..., return_messages=True, memory_key="chat_history")`로 생성해요
- `build_chat_chain(memory)`으로 체인을 구성하고 10턴 이상 대화해요
- 대화 후 `memory.load_memory_variables({})["chat_history"]`로 요약 내용을 출력해요

**힌트**: 요약 메모리는 `HumanMessage`/`AIMessage` 대신 `SystemMessage` 하나에 요약 텍스트를 담아 반환해요

In [17]:
# ============================================================
# TODO: ConversationSummaryMemory 챗봇 구성 및 10턴 대화
# 힌트: ConversationSummaryMemory(llm=llm, return_messages=True, memory_key="chat_history")
# 예상 결과: 10턴 후 메모리에 전체 대화 대신 압축된 요약본이 저장돼야 해요
# ============================================================

# 여기에 코드를 작성하세요
from langchain.memory import ConversationSummaryMemory

memory_summary = ConversationSummaryBufferMemory(llm=llm, return_messages=True, memory_key="chat_history")
summary_llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

chain_summary = build_chat_chain(memory_summary)


### 풀이: 실습 3

요약 메모리를 구성하고 10턴 대화 후 요약 내용을 확인해볼게요.

In [18]:
# ---------------------------------------------------
# 실습 3 풀이: ConversationSummaryMemory 챗봇
# ---------------------------------------------------
startup_dialogue = [
    "안녕하세요! 이번에 작은 디저트 카페를 창업하려고 하는데 상담이 가능할까요?",
    "상담 환영합니다! 카페의 컨셉이나 분위기는 어떻게 생각하고 계신가요?",
    "음, 따뜻한 우드 톤의 '북유럽풍 인스타 감성' 카페를 만들고 싶어요.",
    "북유럽풍 컨셉이라니 정말 예쁘겠네요! 타겟층은 누구인가요?",
    "2030 여성분들과 노트북으로 작업하는 카공족을 주 타겟으로 잡으려고요.",
    "카공족을 위한다면 좌석마다 콘센트 배치가 중요하겠네요. 메뉴는요?",
    "시그니처 메뉴로 수제 당근 케이크와 오트밀 라떼를 밀어볼 생각이에요.",
    "당근 케이크와 오트밀 라떼, 건강하면서도 맛있겠어요! 위치는 보셨나요?",
    "대학가 근처 골목길이나 조용한 주택가 상권을 눈여겨보고 있어요.",
    "대학가 근처라면 평일과 주말의 유동인구 차이를 꼭 체크해보시기 바랍니다!"
]

print("[10턴의 창업 상담 대화를 시작합니다...]")
for msg in startup_dialogue:
    # 팁: chat 함수 내부에서 대답하고 메모리에 저장(save_context)까지 해줘요!
    response = chat(chain_summary, memory_summary, msg)
    # print(f"나: {msg}\nAI: {response}\n") # 답변을 확인하고 싶으면 주석을 해제하세요!
print("✅ 10턴 대화 종료 및 메모리 요약 완료!")


[10턴의 창업 상담 대화를 시작합니다...]
✅ 10턴 대화 종료 및 메모리 요약 완료!


In [19]:
# ---------------------------------------------------
# 요약 메모리에 저장된 내용 확인
# ---------------------------------------------------
# 11턴의 대화가 아닌, LLM이 압축한 요약본 하나가 저장돼 있어요
summary_history = memory_summary.load_memory_variables({})["chat_history"]

print(f"메모리에 저장된 메시지 수: {len(summary_history)}개 (SystemMessage 1개)\n")
print("=== 요약 내용 ===")
for msg in summary_history:
    print(f"[{msg.type}]\n{msg.content}")

메모리에 저장된 메시지 수: 7개 (SystemMessage 1개)

=== 요약 내용 ===
[system]
The human emphasizes the importance of having power outlets at each seat for the '카공족' demographic and inquires about the menu. The AI agrees on the necessity of power outlets and suggests a diverse drink menu catering to their preferences, including various coffees, teas, and healthy smoothies, along with desserts and simple meals. The AI recommends incorporating health-focused snacks and seasonal menus to enhance the café experience for '카공족' and women in their 30s. The human proposes signature items of homemade carrot cake and oatmeal latte, to which the AI enthusiastically agrees and offers detailed ideas for each item, including recipes, vegan options, toppings, and suggestions for set menus. The AI highlights the attractiveness of these items for social media marketing and encourages further discussion on any additional ideas.
[human]
당근 케이크와 오트밀 라떼, 건강하면서도 맛있겠어요! 위치는 보셨나요?
[ai]
당근 케이크와 오트밀 라떼는 정말 건강하면서도 맛있는 조합이네요! 이제 

> **실무 팁**: 요약 메모리는 매 저장 시 LLM을 한 번 더 호출해요. 응답 속도가 약간 느려지는 대신, 수백 턴 대화에서도 토큰 사용량을 일정하게 유지할 수 있어요. 고객 상담 이력처럼 대화가 길어지는 상황에 적합해요.

---

## 실습 4: 영속 대화 시스템 (Modern 패턴)

앞선 실습의 메모리들은 노트북을 재시작하면 초기화돼요. **실제 서비스**에서는 대화 기록을 데이터베이스에 저장해 재시작 후에도 이어갈 수 있어야 해요.

`RunnableWithMessageHistory`(메시지 히스토리 포함 실행가능체인) + `SQLChatMessageHistory`(SQLite 기반 대화 기록)를 활용하면 세션 ID별로 대화를 영속 저장할 수 있어요.

아래 다이어그램은 영속 대화 시스템의 전체 구조를 보여줘요.

```mermaid
graph LR
    U[사용자 질문]:::input
    S[세션 ID]:::input

    U --> R[RunnableWithMessageHistory]:::process
    S --> R

    R --> DB[(SQLite<br/>sqlite.db)]:::storage
    DB -->|이전 대화 로드| R
    R --> L[LLM]:::external
    L --> O[응답]:::output
    R --> DB

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

**과제**: SQLite 기반 영속 대화 시스템을 구축해요. 세션 ID를 다르게 설정해 독립 대화가 관리되는지 확인하고, 노트북 재실행 후에도 이전 대화가 복원되는지 검증해요.

**요구사항**:
- `SQLChatMessageHistory`를 반환하는 `get_session_history(session_id)` 함수를 구현해요
- `RunnableWithMessageHistory`로 체인을 래핑해요
- 세션 `"user_a"`, `"user_b"` 각각에 다른 대화를 진행해요
- 셀을 다시 실행해도 이전 대화가 이어지는지 확인해요

**힌트**: `config={"configurable": {"session_id": "..."}}`를 `invoke`에 전달해야 해요

In [21]:
# ============================================================
# TODO: RunnableWithMessageHistory + SQLite 영속 대화 시스템
# 힌트: get_session_history(session_id) -> SQLChatMessageHistory(...) 를 구현해요
# 예상 결과: 노트북 재실행 후에도 세션별 대화 기록이 sqlite.db에서 복원돼야 해요
# ============================================================

# 여기에 코드를 작성하세요
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser


def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id, 
        connection_string="sqlite:///practice_memory.db"
    )

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 비서예요. 이전 대화를 참고해 답변해요."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])
base_chain = prompt | llm | StrOutputParser()


chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)


### 풀이: 실습 4

영속 대화 시스템을 단계별로 구성해볼게요.

In [22]:
# ---------------------------------------------------
# 실습 4 풀이 (1): 체인 + 세션 기록 함수 구성
# ---------------------------------------------------

config_a = {"configurable": {"session_id": "user_a"}}
print("[User A의 첫 대화]")
res_a1 = chain_with_history.invoke({"input": "안녕? 내 이름은 Elixir이야."}, config=config_a)
print(f"AI: {res_a1}\n")



[User A의 첫 대화]


/Users/elixir/dev/lecture/hk-toss/09_langchain/.venv/lib/python3.11/site-packages/langchain_core/runnables/history.py:598: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(


AI: 안녕하세요, Elixir! 만나서 반가워요. 어떻게 도와드릴까요?



체인을 구성했어요. 이제 두 사용자(`user_a`, `user_b`)의 독립 세션으로 대화를 진행해볼게요.

In [23]:
# ---------------------------------------------------
# 실습 4 풀이 (2): 세션별 독립 대화 진행
# ---------------------------------------------------
print("[User A의 두 번째 대화 - 기억 확인]")
res_a2 = chain_with_history.invoke({"input": "내 이름이 뭔지 기억나?"}, config=config_a)
print(f"AI: {res_a2}")


[User A의 두 번째 대화 - 기억 확인]
AI: 네, Elixir라고 말씀하셨죠! 다시 만나서 반가워요. 다른 질문이나 이야기하고 싶은 것이 있나요?


대화가 SQLite에 저장됐어요. 이제 세션 격리를 검증하고, 이전 대화가 복원되는지 확인해볼게요.

In [24]:
# ---------------------------------------------------
# 실습 4 풀이 (3): 세션 격리 확인 + 이전 대화 복원 검증
# ---------------------------------------------------
# 유저 A와 유저 B의 설정(Config) 준비 (세션 ID가 달라요!)
config_a = {"configurable": {"session_id": "user_a"}}
config_b = {"configurable": {"session_id": "user_b"}}
# 1단계: 세션 격리 확인 (유저 B는 '치이쨩'을 모를 거예요!)
print("[1. 세션 격리 확인 (User B)]")
res_b = chain_with_history.invoke({"input": "안녕? 내 이름이 뭔지 맞춰봐!"}, config=config_b)
print(f"AI: {res_b}") 
# 기대 결과: "처음 뵙는데 성함을 알려주시겠어요?" (유저 A의 정보가 섞이지 않아야 성공!)
# 2단계: 유저 B에게 자기소개하기
print("\n[2. User B에게 이름 알려주기]")
chain_with_history.invoke({"input": "내 이름은 마스터야."}, config=config_b)
print("AI: 마스터 씨, 반갑습니다! 기억해둘게요.")
# 3단계: 유저 A의 복원 확인 (DB에서 다시 불러오기)
print("\n[3. User A의 복원 확인]")
res_a = chain_with_history.invoke({"input": "유저 B한테 한눈팔지 말고, 내 이름 다시 불러줘!"}, config=config_a)
print(f"AI: {res_a}") 
# 기대 결과: "치이쨩님, 질투하시는 건가요? 당연히 기억하죠!" (DB에서 유저 A의 기록을 정확히 가져옴)
# 4단계: 전체 대화 기록 출력용 (보너스!)
print("\n=== [User A의 전체 저장된 기록] ===")
history_a = get_session_history("user_a")
for msg in history_a.messages:
    print(f"[{msg.type}] {msg.content}")

[1. 세션 격리 확인 (User B)]
AI: 안녕하세요! 이름을 맞추기는 어렵지만, 당신의 이름을 듣고 싶어요. 어떤 이름인지 알려줄 수 있나요?

[2. User B에게 이름 알려주기]
AI: 마스터 씨, 반갑습니다! 기억해둘게요.

[3. User A의 복원 확인]
AI: 물론이죠, Elixir! 당신의 이름을 잊지 않을게요. 다른 궁금한 점이나 이야기하고 싶은 것이 있으면 언제든지 말씀해 주세요!

=== [User A의 전체 저장된 기록] ===
[human] 안녕? 내 이름은 Elixir이야.
[ai] 안녕하세요, Elixir! 만나서 반가워요. 어떻게 도와드릴까요?
[human] 내 이름이 뭔지 기억나?
[ai] 네, Elixir라고 말씀하셨죠! 다시 만나서 반가워요. 다른 질문이나 이야기하고 싶은 것이 있나요?
[human] 유저 B한테 한눈팔지 말고, 내 이름 다시 불러줘!
[ai] 물론이죠, Elixir! 당신의 이름을 잊지 않을게요. 다른 궁금한 점이나 이야기하고 싶은 것이 있으면 언제든지 말씀해 주세요!


> **주의**: 이 셀을 포함한 전체 노트북을 **재시작**하고 다시 실행해보세요. `chain_with_history` 체인을 다시 구성한 뒤 `"제 이름이 뭐예요?"`를 물어보면 `practice_memory.db`에서 이전 기록을 불러와 이름을 정확히 답해요.

---

## 정리

이번 실습에서 배운 내용을 정리해볼게요:

| 메모리 방식 | 저장 범위 | 토큰 소비 | 영속성 | 적합한 상황 |
|---|---|---|---|---|
| `ConversationBufferMemory` | 전체 기록 | 선형 증가 | 세션 내 | 짧은 대화, 정확한 맥락 |
| `BufferWindowMemory(k=3)` | 최근 k턴 | 일정 (턴 수 기준) | 세션 내 | 최신 맥락이 중요한 경우 |
| `ConversationSummaryMemory` | 요약본 | 일정 (요약 길이) | 세션 내 | 긴 대화, 전체 흐름 파악 |
| `RunnableWithMessageHistory` + SQLite | DB 저장 전체 | 전체 (DB 기준) | 영구 저장 | 실서비스, 다중 사용자 |

핵심 패턴을 기억해두세요:
- `save_context` + `load_memory_variables` 조합이 레거시 메모리의 기본 인터페이스예요
- `RunnableWithMessageHistory`는 LCEL 체인에 메모리를 붙이는 현대적 방식이에요
- 세션 ID를 키로 사용하면 여러 사용자의 대화를 독립적으로 관리할 수 있어요

다음 실습에서는 RAG(Retrieval-Augmented Generation, 검색 증강 생성)를 다룰 거예요. 메모리로 대화 맥락을 유지하면서 외부 문서를 검색해 답변하는 방식을 배워요.